# Rapport de Synthèse : Analyseur de Signaux Mixtes (Finance & NLP)

## 1. Introduction et Cadrage du Projet

L'objectif principal de ce projet est de concevoir et de déployer un pipeline de Machine Learning "bout-en-bout" (end-to-end) capable de prédire la tendance à court terme d'un actif financier. Concrètement, le système doit déterminer sous forme de classification binaire si le prix de l'action ciblée (Ici celle d'Apple (APPL)) sera en hausse ou en baisse à un horizon de 5 jours.  

La singularité et la valeur ajoutée de ce projet reposent sur l'hybridation des données. Pour imiter au mieux l'analyse d'un trader, le modèle combine deux sources d'informations distinctes :  
* Les signaux quantitatifs : L'analyse des séries temporelles classiques, incluant l'historique des prix et le calcul d'indicateurs techniques.  
* Les signaux qualitatifs : Le Traitement du Langage Naturel (NLP) appliqué quotidiennement à l'actualité financière pour capter la psychologie et le "sentiment" du marché.  

Au-delà de la prédiction pure, l'architecture de ce projet a été pensée pour démontrer une maîtrise complète de la chaîne de valeur de la donnée, incluant :  
* L'Ingénierie des Données (Data Engineering) : Création d'un pipeline ETL automatisé (Extraction via API, Transformation, et Chargement).  
* La Modélisation Statistique : Mise en compétition d'algorithmes de Machine Learning avancés pour résoudre un problème de classification complexe.  
* Le Développement Logiciel : Structuration du code en scripts Python modulaires, gestion de base de données (SQL), et utilisation des bonnes pratiques de versioning (Git). 

## 2. Feature Engineering et Justification des Signaux

La performance d'un modèle repose massivement sur la qualité des variables explicatives qu'on lui fournit. Pour ce pipeline, nous avons conçu un jeu de données hybride capturant à la fois la dynamique intrinsèque du prix (analyse technique) et la perception exogène du marché (analyse de sentiment).  

### 2.1. Signaux Quantitatifs : Dynamique des Prix et Risque

Les indicateurs techniques ont été calculés pour fournir au modèle des informations sur le momentum, la réversion à la moyenne et le niveau de risque.  

* Moyennes Mobiles (SMA_20 et EMA_20) : La moyenne mobile simple (SMA) sur 20 jours fournit une ligne de base de la tendance à court/moyen terme. La moyenne mobile exponentielle (EMA), en accordant un poids plus important aux prix récents, permet au modèle de détecter plus rapidement les ruptures de tendance.
* Volatilité Historique (14 jours) : Cet indicateur mesure l'incertitude du marché. Il est calculé via l'écart-type glissant des rendements quotidiens, annualisé selon la formule : $Volatility = \sigma_{14} \times \sqrt{252}$. Les pics de volatilité précèdent souvent des retournements de marché majeurs, un signal crucial pour les arbres de décision.
* Relative Strength Index (RSI_14) : Il s'agit d'un oscillateur de momentum mesurant la vélocité des mouvements de prix pour identifier les zones de surachat ou de survente. Sa formulation mathématique est :$RSI = 100 - \frac{100}{1 + RS}$où $RS$ (Relative Strength) représente le ratio de la moyenne des gains sur la moyenne des pertes sur 14 jours.

### 2.2. Signaux Qualitatifs : NLP et Psychologie du Marché

Les modèles quantitatifs purs souffrent d'un retard inhérent en ne réagissant qu'aux aux prix passés. L'intégration des données textuelles vise à donner au modèle une capacité d'anticipation basée sur le flux d'informations. 

* Le choix de FinBERT : Les modèles NLP généralistes échouent souvent à saisir les nuances du jargon financier (par exemple, le mot "chute" peut être positif s'il s'agit du taux de chômage). Nous avons implémenté FinBERT, un modèle de type Transformer spécifiquement ré-entraîné par ProsusAI sur un corpus financier massif (Financial PhraseBank).  
* Agrégation Journalière (daily_sentiment et new_volumes) : Les prédictions brutes de FinBERT (probabilités des classes Positif, Négatif, Neutre) ont été pondérées et agrégées par jour. Cela permet de transformer un flux de presse non structuré en une série temporelle continue, parfaitement alignable avec nos données de marché dans la table SQL fact_news_sentiment.

## 3. Évaluation et Performances des Modèles

### 3.1. Méthodologie d'Évaluation et Validation Temporelle

En modélisation financière, l'utilisation d'une validation croisée classique (type K-Fold aléatoire) est à proscrire, car elle entraînerait une fuite de données (data leakage) du futur vers le passé. Pour garantir l'intégrité de nos tests sur l'action Apple, nous avons mis en place plusieurs outils : 
* Un découpage chronologique strict (80% / 20%) : Le modèle s'entraîne uniquement sur le passé lointain et est évalué exclusivement sur les données les plus récentes.  
* Un TimeSeriesSplit (Cross-Validation temporelle) : L'optimisation des hyperparamètres via GridSearchCV a respecté l'ordre chronologique des blocs d'entraînement pour simuler un véritable environnement de trading en conditions réelles.  
* Une formulation en classification binaire : La variable cible indique si le cours de clôture à un horizon de 5 jours sera supérieur ($1$) ou inférieur ($0$) au cours actuel.  

### 3.2. Analyse Comparative : Random Forest vs XGBoost

Les deux algorithmes ensemblistes testés ont révélé des dynamiques d'apprentissage radicalement opposées sur notre jeu de données.

* Le comportement du Random Forest (Modèle Conservateur) : Le Random Forest construit des arbres indépendants en parallèle. Face au bruit inhérent des séries temporelles boursières, la moyenne des votes de la forêt a adopté une stratégie de lissage extrême. Sur l'échantillon de test, le modèle a privilégié la classe majoritaire, démontrant une incapacité à isoler les signaux faibles de retournement de tendance. Bien qu'il cherche à contrer le surapprentissage, sa structure indépendante s'est montrée trop rigide pour capter la non-linéarité des signaux mixtes (prix + sentiment).
* Le comportement de XGBoost (Gradient Boosting Séquentiel) : À l'inverse, XGBoost construit ses arbres de manière itérative, chaque nouvel arbre est spécifiquement entraîné pour corriger les erreurs résiduelles des arbres précédents. Cette approche séquentielle lui a permis de s'adapter finement aux variations de prix et de surperformer, atteignant une précision globale supérieure à celle de Random Forest.

### 3.3.  Analyse des Métriques Avancées (ROC, Precision-Recall et Confusion)

Pour valider la robustesse de XGBoost, l'analyse ne se limite pas à l'exactitude globale (Accuracy) :

* La Matrice de Confusion : Elle met en évidence la capacité du modèle à identifier correctement les zones de Hausse et de Baisse, minimisant les faux signaux par rapport au Random Forest.
* Les Courbes ROC et Precision-Recall : L'analyse des scores de probabilité à travers l'aire sous la courbe (AUC) confirme la robustesse du classifieur, lui conférant une capacité discriminante satisfaisante pour envisager une automatisation des ordres de décision.

### Comparatif : Random Forest vs XGBoost

* **Random Forest :** Ce modèle ensembliste s'est montré trop "conservateur" face au bruit des données synthétiques, finissant par prédire une classe unique (lissant les signaux). 
* **XGBoost :** En construisant ses arbres de manière séquentielle pour corriger ses propres erreurs (Gradient Boosting), XGBoost a réussi à capter les patterns sous-jacents, obtenant une Accuracy supérieure à 70%.

*(Insérez ici une cellule de code pour afficher vos graphiques : Matrice de Confusion XGBoost et Courbes ROC/PR)*

## 4. Importance des Variables et Conclusion Finale

*(Insérez ici une cellule de code pour afficher le graphique Feature Importance de XGBoost)*

**Le sentiment des actualités permet-il réellement d'améliorer les prédictions ?**
L'analyse de l'importance des variables (Feature Importance) montre que... *(à vous de compléter en lisant votre graphique : si daily_sentiment est bien classé, le NLP a apporté de la valeur !)*.

En conclusion, ce projet valide la faisabilité technique d'un pipeline hybride automatisé, tout en soulignant qu'un passage en production réel nécessiterait un historique de données (NLP et Prix) sur plusieurs années pour maximiser la robustesse de l'algorithme.